In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv()

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
llm = init_chat_model(
    model=os.getenv('MOONSHOT_MODEL_FAST'), 
    model_provider='openai',
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url=os.getenv("MOONSHOT_BASE_URL"),
    temperature=0.1
)

In [3]:
llm = llm.with_config(tags=["final_answer"])

In [4]:
from mainGraph import MainGraph

main_graph = MainGraph(llm)
main_graph.compile_main_graph()

## 事件流模式

可以对于LangGraph中的每一node输出进行监听，比起流式监听可以监听到更多信息。

In [5]:
config = {"configurable": {'thread_id': 'thread-1'}}
events = []
async for event in agent.app.astream_events(
    input={'query': "小酥肉味道如何？怎么做？适合吃火锅的时候吃吗？它和鸡米花哪个更适合作为下酒菜？"}, 
    config=config, 
    version='v2'
):
    events.append(event)

/opt/conda/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MultiQueryComposer(analyz...合作为下酒菜吗']), input_type=MultiQueryComposer])
  return self.__pydantic_serializer__.to_python(


['小酥肉味道怎么样', '小酥肉的做法', '小酥肉适合吃火锅吗', '鸡米花适合作为下酒菜吗']
['general', 'detail', 'general', 'general']
子查询: 小酥肉味道怎么样, 查询类型: general
子查询: 小酥肉的做法, 查询类型: detail
子查询: 小酥肉适合吃火锅吗, 查询类型: general
子查询: 鸡米花适合作为下酒菜吗, 查询类型: general


In [6]:
events[0], events[40], events[-1]

({'event': 'on_chain_start',
  'data': {'input': {'query': '小酥肉味道如何？怎么做？适合吃火锅的时候吃吗？它和鸡米花哪个更适合作为下酒菜？'}},
  'name': 'LangGraph',
  'tags': [],
  'run_id': '019d8c25-57ea-7320-943e-84bc27c955d7',
  'metadata': {'thread_id': 'thread-1', 'ls_integration': 'langgraph'},
  'parent_ids': []},
 {'event': 'on_chat_model_stream',
  'data': {'chunk': AIMessageChunk(content='适合', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--019d8c25-57f0-74e1-b742-dc3a7935d005', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])},
  'run_id': '019d8c25-57f0-74e1-b742-dc3a7935d005',
  'name': 'ChatOpenAI',
  'tags': ['seq:step:1', 'final_answer'],
  'metadata': {'thread_id': 'thread-1',
   'ls_integration': 'langchain_chat_model',
   'langgraph_step': 1,
   'langgraph_node': 'multi_query_composer',
   'langgraph_triggers': ('branch:to:multi_query_composer',),
   'langgraph_path': ('__pregel_pull', 'multi_query_composer'),
   'langgraph_checkpoint_ns': 'multi_query_compose

In [20]:
config = {"configurable": {'thread_id': 'thread-1'}}
events = []
async for event in agent.app.astream_events(
    input={'query': "小酥肉味道如何？怎么做？适合吃火锅的时候吃吗？它和鸡米花哪个更适合作为下酒菜？"}, 
    config=config, 
    version='v2'
):
    if event['metadata'].get('langgraph_node') == 'multi_query_composer':
        if event['event'] == 'on_chat_model_stream':
            print(event['data']['chunk'].content, end='', flush=True)

{"analyze": "用户共提出4个问题：1. 小酥肉味道如何（菜品问答）；2. 小酥肉怎么做（菜谱生成）；3. 小酥肉适合吃火锅时吃吗（菜品问答）；4. 小酥肉 vs 鸡米花哪个更适合下酒（菜品问答）。需要拆成4个单菜品、单意图的子问题。", "queries": ["小酥肉味道怎么样", "小酥肉的做法", "小酥肉适合吃火锅的时候吃吗", "小酥肉适合作为下酒菜吗"]}

/opt/conda/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MultiQueryComposer(analyz...合作为下酒菜吗']), input_type=MultiQueryComposer])
  return self.__pydantic_serializer__.to_python(


['小酥肉味道怎么样', '小酥肉的做法', '小酥肉适合吃火锅的时候吃吗', '小酥肉适合作为下酒菜吗']
['general', 'detail', 'general', 'general']
子查询: 小酥肉味道怎么样, 查询类型: general
子查询: 小酥肉的做法, 查询类型: detail
子查询: 小酥肉适合吃火锅的时候吃吗, 查询类型: general
子查询: 小酥肉适合作为下酒菜吗, 查询类型: general


In [14]:
for event in events:
    if event['event'] == 'on_chat_model_stream':
        a = event
        break

## 流式输出模式

可以对每个节点的全局状态进行监听，也可以针对大模型流式返回进行监听

In [33]:
config = {"configurable": {'thread_id': 'thread-1'}}
chunks = []
async for chunk in agent.app.astream(
    input={'query': "小酥肉味道如何？怎么做？适合吃火锅的时候吃吗？它和鸡米花哪个更适合作为下酒菜？"}, 
    config=config, 
    stream_mode="messages", 
    version='v1'
):
    chunk_msg, metadata = chunk
    if metadata.get('langgraph_node') == 'multi_query_composer':
        print(chunk_msg.content, end='', flush=True)

{"analyze": "用户共提出4个问题：1.小酥肉味道如何？2.小酥肉怎么做？3.小酥肉适合吃火锅时吃吗？4.小酥肉和鸡米花哪个更适合做下酒菜？需要拆成4个独立的单菜品/单场景子问题。", "queries": ["小酥肉味道怎么样", "小酥肉的做法", "小酥肉适合吃火锅吗", "小酥肉适合做下酒菜吗"]}

/opt/conda/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MultiQueryComposer(analyz...适合做下酒菜吗']), input_type=MultiQueryComposer])
  return self.__pydantic_serializer__.to_python(


['小酥肉味道怎么样', '小酥肉的做法', '小酥肉适合吃火锅吗', '小酥肉适合做下酒菜吗']
['general', 'detail', 'general', 'general']
子查询: 小酥肉味道怎么样, 查询类型: general
子查询: 小酥肉的做法, 查询类型: detail
子查询: 小酥肉适合吃火锅吗, 查询类型: general
子查询: 小酥肉适合做下酒菜吗, 查询类型: general


## 普通输出模式

直接输出最后state

In [4]:
config = {"configurable": {"thread_id": "thread-2"}}

res = await agent.app.ainvoke(
    MainGraph.MainState(query="小酥肉味道如何？怎么做？适合吃火锅的时候吃吗？它和鸡米花哪个更适合作为下酒菜？"), 
    config=config,
)

['小酥肉味道怎么样', '小酥肉的做法', '小酥肉适合吃火锅吗', '小酥肉适合做下酒菜吗', '鸡米花适合做下酒菜吗']
['general', 'detail', 'general', 'general', 'general']
子查询: 小酥肉味道怎么样, 查询类型: general
子查询: 小酥肉的做法, 查询类型: detail
子查询: 小酥肉适合吃火锅吗, 查询类型: general
子查询: 小酥肉适合做下酒菜吗, 查询类型: general
子查询: 鸡米花适合做下酒菜吗, 查询类型: general


In [5]:
res

{'messages': [],
 'query': '小酥肉味道如何？怎么做？适合吃火锅的时候吃吗？它和鸡米花哪个更适合作为下酒菜？',
 'branch_queries': ['小酥肉味道怎么样',
  '小酥肉的做法',
  '小酥肉适合吃火锅吗',
  '小酥肉适合做下酒菜吗',
  '鸡米花适合做下酒菜吗'],
 'branch_categories': ['general', 'detail', 'general', 'general', 'general'],
 'branch_results': ['子查询: 小酥肉味道怎么样, 查询类型: general',
  '子查询: 小酥肉的做法, 查询类型: detail',
  '子查询: 小酥肉适合吃火锅吗, 查询类型: general',
  '子查询: 小酥肉适合做下酒菜吗, 查询类型: general',
  '子查询: 鸡米花适合做下酒菜吗, 查询类型: general'],
 'result': '子查询: 小酥肉味道怎么样, 查询类型: general | 子查询: 小酥肉的做法, 查询类型: detail | 子查询: 小酥肉适合吃火锅吗, 查询类型: general | 子查询: 小酥肉适合做下酒菜吗, 查询类型: general | 子查询: 鸡米花适合做下酒菜吗, 查询类型: general'}

# LangGraph检查点 Memoryer

In [18]:
agent.app.get_state(config=config).values

{'messages': [],
 'query': '小酥肉味道如何？怎么做？适合吃火锅的时候吃吗？它和鸡米花哪个更适合作为下酒菜？',
 'branch_queries': ['小酥肉味道怎么样',
  '小酥肉的做法',
  '小酥肉适合吃火锅吗',
  '小酥肉适合做下酒菜吗',
  '鸡米花适合做下酒菜吗'],
 'branch_categories': ['general', 'detail', 'general', 'general', 'general'],
 'branch_results': ['子查询: 小酥肉味道怎么样, 查询类型: general',
  '子查询: 小酥肉的做法, 查询类型: detail',
  '子查询: 小酥肉适合吃火锅吗, 查询类型: general',
  '子查询: 小酥肉适合做下酒菜吗, 查询类型: general',
  '子查询: 鸡米花适合做下酒菜吗, 查询类型: general'],
 'result': '子查询: 小酥肉味道怎么样, 查询类型: general | 子查询: 小酥肉的做法, 查询类型: detail | 子查询: 小酥肉适合吃火锅吗, 查询类型: general | 子查询: 小酥肉适合做下酒菜吗, 查询类型: general | 子查询: 鸡米花适合做下酒菜吗, 查询类型: general'}

In [7]:
lis = list(agent.app.checkpointer.list(config))

In [8]:
lis[-1].checkpoint

{'v': 4,
 'ts': '2026-04-13T13:47:53.824434+00:00',
 'id': '1f1373f5-ece5-6731-bfff-2db85b2bde9e',
 'channel_versions': {'__start__': '00000000000000000000000000000001.0.10655891128021033'},
 'versions_seen': {'__input__': {}},
 'updated_channels': ['__start__'],
 'channel_values': {'__start__': {'subquery': '鸡米花适合做下酒菜吗',
   'query_category': 'general'}}}

In [13]:
for t in lis:
    print(t.checkpoint['channel_values'])

{'query': '小酥肉味道如何？怎么做？适合吃火锅的时候吃吗？它和鸡米花哪个更适合作为下酒菜？', 'branch_queries': ['小酥肉味道怎么样', '小酥肉的做法', '小酥肉适合吃火锅吗', '小酥肉适合做下酒菜吗', '鸡米花适合做下酒菜吗'], 'branch_categories': ['general', 'detail', 'general', 'general', 'general'], '__pregel_tasks': [], 'messages': [], 'branch_results': ['子查询: 小酥肉味道怎么样, 查询类型: general', '子查询: 小酥肉的做法, 查询类型: detail', '子查询: 小酥肉适合吃火锅吗, 查询类型: general', '子查询: 小酥肉适合做下酒菜吗, 查询类型: general', '子查询: 鸡米花适合做下酒菜吗, 查询类型: general'], 'result': '子查询: 小酥肉味道怎么样, 查询类型: general | 子查询: 小酥肉的做法, 查询类型: detail | 子查询: 小酥肉适合吃火锅吗, 查询类型: general | 子查询: 小酥肉适合做下酒菜吗, 查询类型: general | 子查询: 鸡米花适合做下酒菜吗, 查询类型: general'}
{'query': '小酥肉味道如何？怎么做？适合吃火锅的时候吃吗？它和鸡米花哪个更适合作为下酒菜？', 'branch_queries': ['小酥肉味道怎么样', '小酥肉的做法', '小酥肉适合吃火锅吗', '小酥肉适合做下酒菜吗', '鸡米花适合做下酒菜吗'], 'branch_categories': ['general', 'detail', 'general', 'general', 'general'], '__pregel_tasks': [], 'messages': [], 'branch_results': ['子查询: 小酥肉味道怎么样, 查询类型: general', '子查询: 小酥肉的做法, 查询类型: detail', '子查询: 小酥肉适合吃火锅吗, 查询类型: general', '子查询: 小酥肉适合做下酒菜吗, 查询类型: general', '